# Cosmos property profiles to Silver - build it with Copilot

This is a guided discovery notebook. Each step gives you a detailed prompt for Fabric's in-notebook Copilot. Paste the prompt into Copilot, place the generated code in the empty cell below it, run the cell, and check the verification note.

Do not open the `Answers` folder unless your facilitator directs you to use the completed reference.

## Step 0 - prepare the workspace

Before writing code:

1. Confirm that `SilverLakehouse` exists with Lakehouse schemas enabled.
2. Confirm that the Cosmos mirrored database is available in the workspace.
3. Record the mirror's Fabric item name. `Property Profiles Mirror` is the suggested name, but your lab may use another name.
4. Open the notebook Copilot pane.

## Step 1 - attach the Silver Lakehouse

**Prompt - paste into Copilot:**

```text
Add a %%configure cell that sets the default lakehouse for this Microsoft Fabric PySpark notebook to the lakehouse named SilverLakehouse. Use the force option so the configuration is applied before the Spark session starts.
```

**Verify:** The generated cell contains `%%configure -f` and names `SilverLakehouse`. Run it before any import or Spark code.

## Step 2 - create portable parameters

**Prompt - paste into Copilot:**

```text
Create a Python parameter cell for this Fabric notebook. Define cosmos_mirror_item with the suggested value "Property Profiles Mirror" and source_table_path with the value "Tables/properties". Add a comment explaining that attendees should change cosmos_mirror_item when the lab uses a different Fabric mirror name, and should include a schema segment in source_table_path if their Cosmos mirror exposes one. Do not add an Azure account name, database name, endpoint, key, or connection string.
```

**Verify:** The cell has only logical Fabric item and table-path parameters. Update `cosmos_mirror_item` if your facilitator supplied another mirror name.

## Step 3 - resolve and read the mirrored Cosmos table

**Prompt - paste into Copilot:**

```text
Write PySpark notebook code that imports requests, notebookutils, and pyspark.sql.functions as F. Read currentWorkspaceId from notebookutils.runtime.context. Create a resolve_item_id(display_name, item_type) function that gets a Fabric API token with notebookutils.credentials.getToken("pbi"), calls the Fabric items API for the current workspace with a Bearer authorization header, checks the HTTP response, requires exactly one item whose displayName equals the requested name, and returns that item's id. Resolve cosmos_mirror_item as item type MirroredDatabase. Build a tenant-neutral OneLake ABFS path using the workspace id, onelake.dfs.fabric.microsoft.com, the resolved mirror id, and source_table_path. Read that path as Delta into a DataFrame named raw, print its schema, and print the row count. Do not hard-code workspace, tenant, Azure account, or database identifiers.
```

**Verify:** `raw.printSchema()` shows `parcelId`, `jurisdiction`, `location`, `site`, `building`, `currentAssessment`, and `inspections`. If the Delta path is not found, inspect the mirror and adjust only `source_table_path`.

## Step 4 - flatten one property profile per parcel

**Prompt - paste into Copilot:**

```text
From the raw Cosmos DataFrame, create a DataFrame named profiles with one row per parcel. Select and rename these fields: parcelId to parcel_id; documentType to document_type; synthetic cast to boolean; neighborhoodId and neighborhoodName to snake_case; jurisdiction.countryCode, regionCode, and currencyCode to country_code, region_code, and currency_code; GeoJSON location.coordinates element 0 cast to double as longitude and element 1 cast to double as latitude; site.syntheticAddress, propertyClassCode, zoningCode, and lotAreaSquareMetres to snake_case with the area cast to double; building.type, yearBuilt, floorAreaSquareMetres, and condition to building_type, year_built, floor_area_square_metres, and building_condition with numeric casts; and currentAssessment.taxYear, assessedValue, and confidenceScore to current_tax_year, current_assessed_value, and confidence_score with numeric casts. Drop rows with null parcel_id or neighborhood_id and de-duplicate by parcel_id. Write profiles in overwrite mode with overwriteSchema true as the managed Delta table dbo.property_profile.
```

**Verify:** `dbo.property_profile` has one row per `parcel_id`. Longitude appears before latitude because the source uses GeoJSON coordinate order.

## Step 5 - explode inspection history

**Prompt - paste into Copilot:**

```text
From raw, create an inspections DataFrame by selecting parcelId as parcel_id and using explode_outer on the inspections array into a struct named inspection. Flatten the struct into inspection_id, inspection_date cast to date, inspection_type, condition, observations, and follow_up_required cast to boolean. Drop rows with null parcel_id or inspection_id and de-duplicate by inspection_id. Write the result in overwrite mode with overwriteSchema true as the managed Delta table dbo.fact_inspection.
```

**Verify:** `dbo.fact_inspection` has one row per inspection and retains the observations array.

## Step 6 - add quality assertions

**Prompt - paste into Copilot:**

```text
Write validation code for profiles and inspections. Assert that no profile has synthetic other than true. Assert that no profile has a null latitude or longitude. Assert that parcel_id is unique in profiles and inspection_id is unique in inspections. Print the profile count, inspection count, and count of inspections where follow_up_required is true. Display five flattened profiles and five inspections for review.
```

**Verify:** Every assertion passes, both managed tables exist under `dbo`, and at least one inspection requires follow-up.

## Step 7 - reflect and compare

Explain why the property profile and inspection fact have different grains. Record one refinement you made to a Copilot response and why. Compare your output table names, columns, and counts with the architecture guide before continuing.